In [1]:
import pandas as pd
import joblib
import stanza
import nltk
import time
from Preprocessing_pipeline import normalize_arabic
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from nltk.corpus import stopwords

# 1. Download models and resources
stanza.download('ar')
nltk.download('stopwords', quiet=True)

# Initialize Stanza and Stopwords
nlp = stanza.Pipeline('ar', processors='tokenize,mwt,pos,lemma', verbose=False)
arabic_stopwords = set(stopwords.words('arabic'))

# 2. Load & Prepare Data
df = pd.read_csv('../data/arabic_reviews.csv')
#df = df.head(1000).copy()
df = df.dropna(subset=['review_description', 'rating']).copy()

# Preprocessing
df['clean_text'] = df['review_description'].apply(
    normalize_arabic,
    nlp=nlp,
    arabic_stopwords=arabic_stopwords
)

def map_sentiment(val):
    val = str(val).strip().lower()
    if val in ['positive', '5', '4', 'ممتاز', 'إيجابي']:
        return 'Positive'
    elif val in ['negative', '1', '2', 'سيء', 'سلبي']:
        return 'Negative'
    return 'Neutral'

df['sentiment'] = df['rating'].apply(map_sentiment)

# 3. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42
)

# 4. Define Classifiers to Compare
classifiers = {
    'Logistic Regression': LogisticRegression(C=2.0, max_iter=1000),
    'Multinomial Naive Bayes': MultinomialNB(alpha=1.0),
    'Linear SVM': LinearSVC(C=1.0)
}

best_model = None
best_accuracy = 0.0
best_model_name = ""

# 5. Evaluate Classifiers
print("\n=== Model Comparison Results ===")
for name, clf in classifiers.items():
    model = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=15000)),
        ('clf', clf)
    ])
    
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n--- {name} ---")
    print(f"Training Time: {train_time:.4f} sec")
    print(f"Accuracy: {acc * 100:.2f}%")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

# 6. Save the Best Model
print(f"\nBest Classifier Selected: {best_model_name} with Accuracy: {best_accuracy * 100:.2f}%")
joblib.dump(best_model, 'Arabic_model_weights.pkl')
print("Best model saved to Arabic_model_weights.pkl")

2026-08-17 02:07:59 INFO: Downloaded file to C:\Users\lenovo\AppData\Local\StanfordNLP\stanza\Cache\1.14.0\resources\resources.json
2026-08-17 02:07:59 INFO: Downloading default packages for language: ar (Arabic) ...
2026-08-17 02:08:01 INFO: File exists: C:\Users\lenovo\AppData\Local\StanfordNLP\stanza\Cache\1.14.0\resources\ar\default.zip
2026-08-17 02:08:05 INFO: Finished downloading models and saved to C:\Users\lenovo\AppData\Local\StanfordNLP\stanza\Cache\1.14.0\resources



=== Model Comparison Results ===

--- Logistic Regression ---
Training Time: 5.6943 sec
Accuracy: 82.83%
              precision    recall  f1-score   support

    Negative       0.80      0.80      0.80      2832
     Neutral       0.21      0.02      0.04       405
    Positive       0.85      0.92      0.88      4772

    accuracy                           0.83      8009
   macro avg       0.62      0.58      0.57      8009
weighted avg       0.80      0.83      0.81      8009


--- Multinomial Naive Bayes ---
Training Time: 2.3316 sec
Accuracy: 83.06%
              precision    recall  f1-score   support

    Negative       0.81      0.80      0.80      2832
     Neutral       0.00      0.00      0.00       405
    Positive       0.84      0.92      0.88      4772

    accuracy                           0.83      8009
   macro avg       0.55      0.57      0.56      8009
weighted avg       0.79      0.83      0.81      8009


--- Linear SVM ---
Training Time: 3.9898 sec
Accuracy: 